In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_all = pd.read_csv("/home/jupyter/workspace/WORKSPACE_BUCKET/data/Bacterial Infections(Replicated + Significant - All).csv")
data_all.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, LogLocator, NullFormatter


# ============================================================
# Settings
# ============================================================

FIGSIZE = (28, 20)

NDD_ORDER = [
    "AD",
    "PD",
    "Dementia",
    "Vascular Dementia"
]

COHORT_ORDER = [
    "UKB",
    "FinnGen",
    "AoU"
]

COLOR_MAP = {
    "AD": "#EEBEC6",
    "PD": "#A8D1D1",
    "Dementia": "#9EA1D4",
    "Vascular Dementia": "#FD8A8A"
}

# Shared log-scaled HR axis
XMIN = 0.75
XMAX = 20

XTICKS = [0.75, 1, 1.5, 2, 3, 5, 10, 20]
XTICK_LABELS = ["0.75", "1", "1.5", "2", "3", "5", "10", "20"]

POINT_SIZE = 180
Y_LABEL_SIZE = 17
GROUP_GAP = 0.45


# ============================================================
# Prepare data
# ============================================================

plot_df = data_all.copy()

# Rename internally so the plot uses HR only
if "HR" not in plot_df.columns:
    if "OR/HR" in plot_df.columns:
        plot_df = plot_df.rename(columns={"OR/HR": "HR"})
    else:
        raise KeyError("Expected a column named 'HR'.")

required_cols = ["NDD", "Description", "Cohort", "HR", "CI Min", "CI Max"]
missing_cols = [col for col in required_cols if col not in plot_df.columns]

if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

for col in ["HR", "CI Min", "CI Max"]:
    plot_df[col] = pd.to_numeric(plot_df[col])

if (plot_df[["HR", "CI Min", "CI Max"]] <= 0).any().any():
    raise ValueError("HR, CI Min, and CI Max must be positive for a log-scaled x-axis.")

if ((plot_df["CI Min"] > plot_df["HR"]) | (plot_df["CI Max"] < plot_df["HR"])).any():
    raise ValueError("Each CI must contain the HR value.")

plot_df["_row_order"] = np.arange(len(plot_df))

plot_df["_cohort_order"] = plot_df["Cohort"].map(
    {cohort: i for i, cohort in enumerate(COHORT_ORDER)}
).fillna(999)


# ============================================================
# Helper functions
# ============================================================

def one_line_text(value):
    return " ".join(str(value).split())


def style_axis(ax):
    ax.set_xscale("log")
    ax.set_xlim(XMIN, XMAX)

    ax.xaxis.set_major_locator(FixedLocator(XTICKS))
    ax.xaxis.set_major_formatter(FixedFormatter(XTICK_LABELS))

    ax.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )
    ax.xaxis.set_minor_formatter(NullFormatter())

    ax.axvline(
        1,
        color="red",
        linestyle="--",
        linewidth=1,
        alpha=0.9,
        zorder=1
    )

    ax.grid(
        axis="x",
        which="major",
        alpha=0.18,
        linewidth=0.8
    )

    ax.tick_params(
        axis="x",
        which="major",
        bottom=True,
        labelbottom=True,
        length=4,
        width=0.8,
        labelsize=25
    )

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=True,
        labelbottom=False,
        length=2.5,
        width=0.6
    )

    ax.tick_params(
        axis="y",
        length=0,
        labelsize=Y_LABEL_SIZE
    )

    ax.set_axisbelow(True)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

    ax.spines["bottom"].set_color("0.25")
    ax.spines["bottom"].set_linewidth(0.8)


def build_grouped_y_axis(df):
    """
    Create y positions where each Description appears once as a group header,
    followed by cohort-only rows.
    """
    y_ticks = []
    y_labels = []
    header_y_values = []

    point_rows = []
    y = 0.0

    descriptions = (
        df.sort_values("_row_order")["Description"]
        .drop_duplicates()
        .tolist()
    )

    for description in descriptions:
        group = (
            df[df["Description"] == description]
            .sort_values("_cohort_order")
            .copy()
        )

        # Description header row
        y_ticks.append(y)
        y_labels.append(one_line_text(description))
        header_y_values.append(y)
        y += 1

        # Cohort rows
        for _, row in group.iterrows():
            row_dict = row.to_dict()
            row_dict["y"] = y
            point_rows.append(row_dict)

            y_ticks.append(y)
            y_labels.append(f"   {row['Cohort']}")
            y += 1

        y += GROUP_GAP

    point_df = pd.DataFrame(point_rows)

    return point_df, y_ticks, y_labels, header_y_values


def plot_panel(ax, df, ndd):
    style_axis(ax)

    display_names ={
     "AD": "Alzheimer's Disease",
     "PD": "Parkinson's Disease",
     "Dementia": "Dementia",
     "Vascular Dementia": "Vascular Dementia",



    }
    ax.set_title(
        display_names.get(ndd,ndd),
        loc="left",
        fontsize=24,
        fontweight="bold",
        pad=22
    )

    df = df.copy()

    if df.empty:
        ax.set_yticks([])
        ax.text(
            0.5,
            0.5,
            "No data",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=10,
            color="dimgray"
        )
        return

    point_df, y_ticks, y_labels, header_y_values = build_grouped_y_axis(df)

    lower_err = point_df["HR"] - point_df["CI Min"]
    upper_err = point_df["CI Max"] - point_df["HR"]

    color = COLOR_MAP.get(ndd, "gray")

    ax.errorbar(
        point_df["HR"],
        point_df["y"],
        xerr=np.vstack([lower_err, upper_err]),
        fmt="none",
        ecolor=color,
        elinewidth=4,
        capsize=6,
        alpha=0.9,
        zorder=2
    )

    ax.scatter(
        point_df["HR"],
        point_df["y"],
        s=POINT_SIZE,
        color=color,
        edgecolors="black",
        linewidth=0.7,
        alpha=0.95,
        zorder=3
    )

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

    # Put first group at top
    ax.set_ylim(max(y_ticks) + 0.6, -0.6)

    # Make Description header labels stand out
    for label, y_value in zip(ax.get_yticklabels(), y_ticks):
        if y_value in header_y_values:
            label.set_fontweight("bold")
            label.set_color("dimgray")
        else:
            label.set_color("black")


# ============================================================
# Build 4-panel figure
# ============================================================

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=FIGSIZE
)

panel_axes = {
    "AD": axes[0, 0],
    "PD": axes[0, 1],
    "Dementia": axes[1, 0],
    "Vascular Dementia": axes[1, 1]
}

for ndd in NDD_ORDER:
    panel_df = plot_df[plot_df["NDD"] == ndd].copy()
    plot_panel(panel_axes[ndd], panel_df, ndd)


# ============================================================
# Final formatting
# ============================================================

fig.supxlabel(
    "Hazard Ratio (log scale), 95% CI",
    fontsize=30,
    y=0.04
)

plt.subplots_adjust(
    left=0.24,
    right=0.99,
    top=0.95,
    bottom=0.08,
    wspace=0.90,
    hspace=0.20
)

plt.show()

# To save:
fig.savefig("/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Plots/forest_plot_data_all.png", dpi=600, transparent = True, bbox_inches="tight")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, LogLocator, NullFormatter

# subset of lag results for poster 

df = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/lags_simplified.csv')


# ============================================================
# Settings
# ============================================================

FIGSIZE = (22, 11)

# Only plot these two panels
NDD_ORDER = [
    "AD",
    "Vascular Dementia"
]

COHORT_ORDER = [
    "FinnGen",
    "AoU"
]

COLOR_MAP = {
    "AD": "#EEBEC6",
    "PD": "#A8D1D1",
    "Dementia": "#9EA1D4",
    "Vascular Dementia": "#FD8A8A"
}

# Shared log-scaled HR axis
XMIN = 0.5
XMAX = 40

XTICKS = [0.5, 0.75, 1, 1.5, 2, 3, 5, 10, 20, 30]
XTICK_LABELS = ["0.5", "0.75", "1", "1.5", "2", "3", "5", "10", "20", "30"]

# Poster-friendly sizing
POINT_SIZE = 180
Y_LABEL_SIZE = 17
TITLE_SIZE = 18
XLABEL_SIZE = 35
XTICK_SIZE = 20
GROUP_GAP = 0.7


# ============================================================
# Prepare data
# ============================================================

plot_df = df.copy()

# Rename internally so the plot uses HR only
if "HR" not in plot_df.columns:
    if "OR/HR" in plot_df.columns:
        plot_df = plot_df.rename(columns={"OR/HR": "HR"})
    else:
        raise KeyError("Expected a column named 'HR'.")

required_cols = ["NDD", "Bacterial Infection", "LAG", "Cohort", "HR", "CI Min", "CI Max"]
missing_cols = [col for col in required_cols if col not in plot_df.columns]

if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

for col in ["HR", "CI Min", "CI Max", "LAG"]:
    plot_df[col] = pd.to_numeric(plot_df[col])

if (plot_df[["HR", "CI Min", "CI Max", "LAG"]] <= 0).any().any():
    raise ValueError("HR, CI Min, and CI Max must be positive for a log-scaled x-axis.")

if ((plot_df["CI Min"] > plot_df["HR"]) | (plot_df["CI Max"] < plot_df["HR"])).any():
    raise ValueError("Each CI must contain the HR value.")

plot_df["_row_order"] = np.arange(len(plot_df))

plot_df["_cohort_order"] = plot_df["Cohort"].map(
    {cohort: i for i, cohort in enumerate(COHORT_ORDER)}
).fillna(999)


# ============================================================
# Helper functions
# ============================================================

def one_line_text(value):
    return " ".join(str(value).split())


def style_axis(ax):
    ax.set_xscale("log")
    ax.set_xlim(XMIN, XMAX)

    ax.xaxis.set_major_locator(FixedLocator(XTICKS))
    ax.xaxis.set_major_formatter(FixedFormatter(XTICK_LABELS))

    ax.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )
    ax.xaxis.set_minor_formatter(NullFormatter())

    ax.axvline(
        1,
        color="red",
        linestyle="--",
        linewidth=1.2,
        alpha=0.9,
        zorder=1
    )

    ax.grid(
        axis="x",
        which="major",
        alpha=0.18,
        linewidth=0.9
    )

    ax.tick_params(
        axis="x",
        which="major",
        bottom=True,
        labelbottom=True,
        length=5,
        width=0.9,
        labelsize=XTICK_SIZE
    )

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=True,
        labelbottom=False,
        length=3,
        width=0.7
    )

    ax.tick_params(
        axis="y",
        length=0,
        labelsize=Y_LABEL_SIZE
    )

    ax.set_axisbelow(True)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

    ax.spines["bottom"].set_color("0.25")
    ax.spines["bottom"].set_linewidth(0.9)


def build_grouped_y_axis(df):
    """
    Create y positions where each Bacterial Infection appears once as a group header,
    followed by cohort-only rows.
    """
    y_ticks = []
    y_labels = []
    header_y_values = []

    point_rows = []
    y = 0.0

    descriptions = (
        df.sort_values("_row_order")["Bacterial Infection"]
        .drop_duplicates()
        .tolist()
    )

    for description in descriptions:
        group = (
            df[df["Bacterial Infection"] == description]
            .sort_values(["LAG", "_cohort_order"])
            .copy()
        )

        # Description header row
        y_ticks.append(y)
        y_labels.append(one_line_text(description))
        header_y_values.append(y)
        y += 1

        # Cohort rows
        for _, row in group.iterrows():
            row_dict = row.to_dict()
            row_dict["y"] = y
            point_rows.append(row_dict)

            y_ticks.append(y)
            lag = int(row["LAG"]) if float(row["LAG"]).is_integer() else row["LAG"]
            y_labels.append(f"   {row['Cohort']}   {lag} yr")
            y += 1

            y += GROUP_GAP

    point_df = pd.DataFrame(point_rows)
    return point_df, y_ticks, y_labels, header_y_values


def plot_panel(ax, df, ndd):
    style_axis(ax)

    display_names = {
        "AD": "Alzheimer's Disease",
        "PD": "Parkinson's Disease",
        "Dementia": "Dementia",
        "Vascular Dementia": "Vascular Dementia",
    }

    ax.set_title(
        display_names.get(ndd, ndd),
        loc="left",
        fontsize=TITLE_SIZE,
        fontweight="bold",
        pad=18
    )

   

    df = df.copy()

    if df.empty:
        ax.set_yticks([])
        ax.text(
            0.5,
            0.5,
            "No data",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=14,
            color="dimgray"
        )
        return

    point_df, y_ticks, y_labels, header_y_values = build_grouped_y_axis(df)

    lower_err = point_df["HR"] - point_df["CI Min"]
    upper_err = point_df["CI Max"] - point_df["HR"]

    color = COLOR_MAP.get(ndd, "gray")

    ax.errorbar(
        point_df["HR"],
        point_df["y"],
        xerr=np.vstack([lower_err, upper_err]),
        fmt="none",
        ecolor=color,
        elinewidth=2.6,
        capsize=5,
        alpha=0.9,
        zorder=2
    )

    ax.scatter(
        point_df["HR"],
        point_df["y"],
        s=POINT_SIZE,
        color=color,
        edgecolors="black",
        linewidth=0.8,
        alpha=0.95,
        zorder=3
    )

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

    # Put first group at top
    ax.set_ylim(max(y_ticks) + 0.8, -0.8)

    # Make infection header labels stand out
    for label, y_value in zip(ax.get_yticklabels(), y_ticks):
        if y_value in header_y_values:
            label.set_fontweight("bold")
            label.set_color("dimgray")
        else:
            label.set_color("black")


# ============================================================
# Build 2-panel figure
# ============================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=FIGSIZE
)

panel_axes = {
    "AD": axes[0],
    "Vascular Dementia": axes[1]
}

for ndd in NDD_ORDER:
    panel_df = plot_df[plot_df["NDD"] == ndd].copy()
    plot_panel(panel_axes[ndd], panel_df, ndd)


# ============================================================
# Final formatting
# ============================================================

fig.supxlabel(
    "Hazard Ratio (log scale) by Lag (years), 95% CI",
    fontsize=25,
    y=0.05
)

plt.subplots_adjust(
    left=0.25,
    right=0.98,
    top=0.90,
    bottom=0.12,
    wspace=0.42
)

plt.show()
# To save:
fig.savefig("/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Plots/forest_plot_lags_simplified", dpi=300, transparent = True, bbox_inches="tight")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, LogLocator, NullFormatter


df = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/lags.csv')

# ============================================================
# Settings
# ============================================================

FIGSIZE = (15, 10.5)

NDD_ORDER = [
    "AD",
    "PD",
    "Dementia",
    "Vascular Dementia"
]

COHORT_ORDER = [
    "FinnGen",
    "AoU"
]

COLOR_MAP = {
    "AD": "#EEBEC6",
    "PD": "#A8D1D1",
    "Dementia": "#9EA1D4",
    "Vascular Dementia": "#FD8A8A"
}

# Shared log-scaled HR axis
XMIN = 0.5
XMAX = 40

XTICKS = [0.5,0.75, 1, 1.5, 2, 3, 5, 10, 20, 30]
XTICK_LABELS = ["0.5","0.75","1","1.5","2","3","5","10","20","30"]
POINT_SIZE = 55
Y_LABEL_SIZE = 6
GROUP_GAP = 0.45


# ============================================================
# Prepare data
# ============================================================

plot_df = df.copy()



# Rename internally so the plot uses HR only
if "HR" not in plot_df.columns:
    if "OR/HR" in plot_df.columns:
        plot_df = plot_df.rename(columns={"OR/HR": "HR"})
    else:
        raise KeyError("Expected a column named 'HR'.")

required_cols = ["NDD", "Bacterial Infection", "LAG", "Cohort", "HR", "CI Min", "CI Max"]
missing_cols = [col for col in required_cols if col not in plot_df.columns]

if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

for col in ["HR", "CI Min", "CI Max", "LAG"]:
    plot_df[col] = pd.to_numeric(plot_df[col])

if (plot_df[["HR", "CI Min", "CI Max", "LAG"]] <= 0).any().any():
    raise ValueError("HR, CI Min, and CI Max must be positive for a log-scaled x-axis.")

if ((plot_df["CI Min"] > plot_df["HR"]) | (plot_df["CI Max"] < plot_df["HR"])).any():
    raise ValueError("Each CI must contain the HR value.")

plot_df["_row_order"] = np.arange(len(plot_df))

plot_df["_cohort_order"] = plot_df["Cohort"].map(
    {cohort: i for i, cohort in enumerate(COHORT_ORDER)}
).fillna(999)


# ============================================================
# Helper functions
# ============================================================

def one_line_text(value):
    return " ".join(str(value).split())


def style_axis(ax):
    ax.set_xscale("log")
    ax.set_xlim(XMIN, XMAX)

    ax.xaxis.set_major_locator(FixedLocator(XTICKS))
    ax.xaxis.set_major_formatter(FixedFormatter(XTICK_LABELS))

    ax.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )
    ax.xaxis.set_minor_formatter(NullFormatter())

    ax.axvline(
        1,
        color="red",
        linestyle="--",
        linewidth=1,
        alpha=0.9,
        zorder=1
    )

    ax.grid(
        axis="x",
        which="major",
        alpha=0.18,
        linewidth=0.8
    )

    ax.tick_params(
        axis="x",
        which="major",
        bottom=True,
        labelbottom=True,
        length=4,
        width=0.8,
        labelsize=9
    )

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=True,
        labelbottom=False,
        length=2.5,
        width=0.6
    )

    ax.tick_params(
        axis="y",
        length=0,
        labelsize=Y_LABEL_SIZE
    )

    ax.set_axisbelow(True)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

    ax.spines["bottom"].set_color("0.25")
    ax.spines["bottom"].set_linewidth(0.8)


def build_grouped_y_axis(df):
    """
    Create y positions where each Description appears once as a group header,
    followed by cohort-only rows.
    """
    y_ticks = []
    y_labels = []
    header_y_values = []

    point_rows = []
    y = 0.0

    descriptions = (
        df.sort_values("_row_order")["Bacterial Infection"]
        .drop_duplicates()
        .tolist()
    )

    for description in descriptions:
        group = (
        df[df["Bacterial Infection"] == description].sort_values(["LAG", "_cohort_order"]) .copy()
    )

        # Description header row
        y_ticks.append(y)
        y_labels.append(one_line_text(description))
        header_y_values.append(y)
        y += 1

        # Cohort rows
        for _, row in group.iterrows():
            row_dict = row.to_dict()
            row_dict["y"] = y
            point_rows.append(row_dict)

            y_ticks.append(y)
            lag = int(row["LAG"]) if float(row["LAG"]).is_integer() else row["LAG"]
            y_labels.append(f"   {row['Cohort']}   {lag}") 
            y += 1
            y += GROUP_GAP
    point_df = pd.DataFrame(point_rows)

    return point_df, y_ticks, y_labels, header_y_values


def plot_panel(ax, df, ndd):
    style_axis(ax)

    display_names = {
    "AD": "Alzheimer's Disease",
    "PD": "Parkinson's Disease",
    "Dementia": "Dementia",
    "Vascular Dementia": "Vascular Dementia",
     }


    ax.set_title(
        display_names.get(ndd,ndd),
        loc="left",
        fontsize=14,
        fontweight="bold",
        pad=22
    )

    df = df.copy()

    if df.empty:
        ax.set_yticks([])
        ax.text(
            0.5,
            0.5,
            "No data",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=10,
            color="dimgray"
        )
        return

    point_df, y_ticks, y_labels, header_y_values = build_grouped_y_axis(df)

    lower_err = point_df["HR"] - point_df["CI Min"]
    upper_err = point_df["CI Max"] - point_df["HR"]

    color = COLOR_MAP.get(ndd, "gray")

    ax.errorbar(
        point_df["HR"],
        point_df["y"],
        xerr=np.vstack([lower_err, upper_err]),
        fmt="none",
        ecolor=color,
        elinewidth=2.2,
        capsize=4,
        alpha=0.85,
        zorder=2
    )

    ax.scatter(
        point_df["HR"],
        point_df["y"],
        s=POINT_SIZE,
        color=color,
        edgecolors="black",
        linewidth=0.7,
        alpha=0.95,
        zorder=3
    )

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

    # Put first group at top
    ax.set_ylim(max(y_ticks) + 0.6, -0.6)

    # Make Description header labels stand out
    for label, y_value in zip(ax.get_yticklabels(), y_ticks):
        if y_value in header_y_values:
            label.set_fontweight("bold")
            label.set_color("dimgray")
        else:
            label.set_color("black")


# ============================================================
# Build 4-panel figure
# ============================================================

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=FIGSIZE
)

panel_axes = {
    "AD": axes[0, 0],
    "PD": axes[0, 1],
    "Dementia": axes[1, 0],
    "Vascular Dementia": axes[1, 1]
}

for ndd in NDD_ORDER:
    panel_df = plot_df[plot_df["NDD"] == ndd].copy()
    plot_panel(panel_axes[ndd], panel_df, ndd)


# ============================================================
# Final formatting
# ============================================================

fig.supxlabel(
    "Hazard Ratio (log scale) by Lag (years), 95% CI",
    fontsize=12,
    y=0.04
)

plt.subplots_adjust(
    left=0.20,
    right=0.98,
    top=0.93,
    bottom=0.09,
    wspace=0.65,
    hspace=0.42
)

plt.show()

# To save:
fig.savefig("/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Plots/forest_plot_all_lags", dpi=300, transparent = True, bbox_inches="tight")